# Arize Phoenix: The Modern Alternative to Evidently Dashboard

Arize Phoenix provides an interactive UI for ML observability, similar to Evidently AI but with a focus on modern workflows and a very intuitive dashboard.

In [ ]:
import pandas as pd
import phoenix as px
from phoenix.session.evaluation import get_qa_with_reference
import numpy as np

## 1. Load Data
We use the reference data created in the baseline model.

In [ ]:
reference_df = pd.read_parquet('../01/data/reference.parquet')

# Simulate 'current' data with some drift
current_df = reference_df.copy()
current_df['trip_distance'] = current_df['trip_distance'] * np.random.uniform(1.2, 1.5, size=len(current_df))
current_df['prediction'] = current_df['prediction'] * 1.1

print(f"Reference shape: {reference_df.shape}")
print(f"Current shape: {current_df.shape}")

## 2. Launch Phoenix Dashboard
This will start a local server and provide a link to the dashboard.

In [ ]:
session = px.launch_app()
print(f"Dashboard URL: {session.url}")

## 3. Log Data for Monitoring
We define our schema and log the data to the Phoenix session.

In [ ]:
from phoenix.trace import Schema
from phoenix.session.evaluation import Action

# Define features to monitor
features = ["passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge"]

schema = px.Schema(
    prediction_column_name="prediction",
    actual_column_name="duration_min",
    feature_column_names=features
)

reference_ds = px.Inferences(dataframe=reference_df, schema=schema, name="reference")
current_ds = px.Inferences(dataframe=current_df, schema=schema, name="current")

px.Client().log_inferences(current_ds)
px.Client().log_inferences(reference_ds)